In [ ]:
import os
import sys
import time
from datetime import datetime
import numpy as np
import matplotlib.pyplot as plt
import warnings
import tensorflow as tf
import pandas as pd

tf.get_logger().setLevel("ERROR")
tf.autograph.set_verbosity(0)

ROOT = os.path.abspath("..")
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

import run_unet_mlp_uv15 as base
import run_unet_mlp_bottleneckvqc_uv15_randomsplit as exp

from functions.nb_helpers import (
    signed_log1p,
    transform_y_signed_log1p,
    repo_root,
    resolve_extracted_uv_dir,
    build_models,
    existing_layers,
    build_multi_output_model,
    spatial_flatten,
    spectrum_and_metrics_from_Z,
    hierarchical_outer_ci,
    inner_bootstrap_metrics,
)

# Back-compat aliases
_as_list = lambda x: x if isinstance(x, list) else [x]

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=UserWarning)


In [ ]:
# Config
height_m = 15.0
last_k = 12
epochs = 1000
batch_size = 2
cond_emb_dim = 16
n_qubits = 5
n_layers = 2
seed = 7
train_frac = 0.8
val_frac = 0.1

base.set_seeds(seed)


In [ ]:
here = str(repo_root())
os.chdir(here)

extracted_uv_dir = str(resolve_extracted_uv_dir(here))

cases = base.list_cases(extracted_uv_dir)

xs, cs, ys, meta = [], [], [], []
for c in cases:
    m_building, u_mean, v_mean = base.load_uv_steady_mean(c.path, height_m=height_m, last_k=last_k)
    x = m_building[..., None].astype(np.float32)
    y = np.stack([u_mean, v_mean], axis=-1).astype(np.float32)
    xs.append(x)
    cs.append(base.cond_vector(c.speed, c.angle_deg))
    ys.append(y)
    meta.append({"file": os.path.basename(c.path), "speed": c.speed, "d_code": c.d_code, "angle_deg": c.angle_deg})

X = np.stack(xs, axis=0)
C = np.stack(cs, axis=0)
Y = np.stack(ys, axis=0)

X.shape, C.shape, Y.shape


In [ ]:
# Non-building u_mean / v_mean distributions
u_all = Y[..., 0]
v_all = Y[..., 1]

valid_mask = (u_all != base.MISSING_VALUE) & (v_all != base.MISSING_VALUE)
u_valid = u_all[valid_mask]
v_valid = v_all[valid_mask]

# Signed log1p transform: x -> sign(x) * log1p(|x|)

u_log = signed_log1p(u_valid)
v_log = signed_log1p(v_valid)

# Train targets in signed-log1p space (buildings keep missing value)
Y_log = transform_y_signed_log1p(Y, base.MISSING_VALUE)


In [ ]:
# random split
if train_frac + val_frac >= 1.0:
    raise ValueError(f"train_frac+val_frac must be < 1. Got {train_frac+val_frac}")

split_idx = exp._random_split_indices(n=len(cases), train_frac=train_frac, val_frac=val_frac, seed=seed)
tr, va, te = split_idx["train"], split_idx["val"], split_idx["test"]

# Condition scaling (fit on train only)
c_scaler = base.StandardScaler()
C_tr = c_scaler.fit_transform(C[tr])
C_va = c_scaler.transform(C[va])
C_te = c_scaler.transform(C[te])

# Output normalization in signed-log1p space (fit on train only, ignoring missing)
y_mean, y_std = base.compute_y_norm_stats(Y_log[tr])
Y_tr = base.normalize_y(Y_log[tr], y_mean, y_std)
Y_va = base.normalize_y(Y_log[va], y_mean, y_std)
Y_te = base.normalize_y(Y_log[te], y_mean, y_std)

X_tr, X_va, X_te = X[tr], X[va], X[te]
len(tr), len(va), len(te)


# C-QB-UNet

In [ ]:
# load pretrained weights
save_dir = os.path.join(here, "checkpoints", "bottleneckvqc_unet_log1p")
os.makedirs(save_dir, exist_ok=True)
best_weights_path = os.path.join(save_dir, "best_val_weights_5q2l.h5")

# Reload model and stats (rebuild the same architecture, then load weights)
model = exp.build_unet_cond_mlp_bottleneck_vqc(
    input_shape=X_tr.shape[1:],
    cond_dim=C_tr.shape[-1],
    cond_emb_dim=cond_emb_dim,
    n_qubits=n_qubits,
    n_layers=n_layers,
)

model.compile(
    optimizer=base.tf.keras.optimizers.Adam(1e-3),
    loss=base.masked_mse_with_grad,
    metrics=[base.masked_mae_metric, base.masked_rmse_metric, base.MaskedR2()],
)

model.load_weights(best_weights_path)


# C-UNet

In [ ]:
# load pretrained weights
save_dir = os.path.join(here, "checkpoints", "mlp_unet_log1p")
os.makedirs(save_dir, exist_ok=True)
best_weights_path = os.path.join(save_dir, "best_val_weights.h5")

# Reload model and stats (rebuild the same architecture, then load weights)
model_mlp = base.build_unet_cond(input_shape=X_tr.shape[1:], cond_dim=C_tr.shape[-1])

model_mlp.compile(
    optimizer=base.tf.keras.optimizers.Adam(1e-3),
    loss=base.masked_mse_with_grad,
    metrics=[base.masked_mae_metric, base.masked_rmse_metric, base.MaskedR2()],
)

model_mlp.load_weights(best_weights_path)


# Channel-wise Effective Dimensions

In [ ]:
# Load-only full-depth spectrum + nested bootstrap (VQC/MLP + Delta)
# This block takes ~15 hrs to run, so we saved the results under `checkpoints/hier_boot_block_1000epochs/_spectral_exports_bootstrap/20260724_214915` so you can directly run the next cell

# ========= Config =========
SEEDS = [0, 7, 11, 19, 23, 29, 31, 42, 123, 2024]
CKPT_ROOT = os.path.join(here, "checkpoints", "hier_boot_block_1000epochs")

B_INNER = 200
B_OUTER = 5000
CI_LEVEL = 0.95
OUTER_SEED = 123

BLOCK_LAYER_SPECS = {
    "down1": {"mlp": ["down1_c1", "down1_c2"], "vqc": ["down1_c1", "down1_c2"]},
    "down2": {"mlp": ["down2_c1", "down2_c2"], "vqc": ["down2_c1", "down2_c2"]},
    "down3": {"mlp": ["down3_c1", "down3_c2"], "vqc": ["down3_c1", "down3_c2"]},
    "down4": {"mlp": ["down4_c1", "down4_c2"], "vqc": ["down4_c1", "down4_c2"]},
    "bottleneck": {"mlp": ["bottleneck_proj", "bottleneck_c2"], "vqc": ["bottleneck_proj"]},
    "up4": {"mlp": ["up_block4_c1", "up_block4_c2"], "vqc": ["up_block4_c1", "up_block4_c2"]},
    "up3": {"mlp": ["up_block3_c1", "up_block3_c2"], "vqc": ["up_block3_c1", "up_block3_c2"]},
    "up2": {"mlp": ["up_block2_c1", "up_block2_c2"], "vqc": ["up_block2_c1", "up_block2_c2"]},
    "up1": {"mlp": ["up_block1_c1", "up_block1_c2"], "vqc": ["up_block1_c1", "up_block1_c2"]},
    "uv_out": {"mlp": ["uv_out"], "vqc": ["uv_out"]},
}
BLOCK_ORDER = list(BLOCK_LAYER_SPECS.keys())

# ========= Helpers =========

# ========= Main loop =========
X_all = X.astype(np.float32)
C_all = c_scaler.transform(C).astype(np.float32)

run_tag = datetime.now().strftime("%Y%m%d_%H%M%S")
save_dir = os.path.join(CKPT_ROOT, "_spectral_exports_bootstrap", run_tag)
os.makedirs(save_dir, exist_ok=True)

spectra_rows = []      # eigenvalue series per seed/block/model
metric_rows = []       # point estimates (per seed/block/model)
# inner store: inner_store[block][who][metric][seed] = arr(B_INNER)
inner_store = {}

t0 = time.time()
for i, seed in enumerate(SEEDS, 1):
    print(f"\n[{i}/{len(SEEDS)}] seed={seed} load & compute")

    tf.keras.backend.clear_session()
    model_vqc, model_mlp = build_models(
        input_shape=X_tr.shape[1:],
        cond_dim=C_tr.shape[-1],
        cond_emb_dim=cond_emb_dim,
        n_qubits=n_qubits,
        n_layers=n_layers,
    )

    vqc_w = os.path.join(CKPT_ROOT, f"seed_{seed}", "vqc", "best_val_weights.h5")
    mlp_w = os.path.join(CKPT_ROOT, f"seed_{seed}", "mlp", "best_val_weights.h5")
    if not os.path.exists(vqc_w):
        print(f"  skip seed={seed}: missing {vqc_w}")
        continue
    if not os.path.exists(mlp_w):
        print(f"  skip seed={seed}: missing {mlp_w}")
        continue

    model_vqc.load_weights(vqc_w)
    model_mlp.load_weights(mlp_w)

    all_mlp, all_vqc = [], []
    for b, spec in BLOCK_LAYER_SPECS.items():
        all_mlp += spec["mlp"]
        all_vqc += spec["vqc"]
    all_mlp = list(dict.fromkeys(all_mlp))
    all_vqc = list(dict.fromkeys(all_vqc))

    all_mlp = existing_layers(model_mlp, all_mlp)
    all_vqc = existing_layers(model_vqc, all_vqc)

    fm_mlp, used_mlp = build_multi_output_model(model_mlp, all_mlp, f"mlp_s{seed}")
    fm_vqc, used_vqc = build_multi_output_model(model_vqc, all_vqc, f"vqc_s{seed}")

    pred_mlp = _as_list(fm_mlp.predict({"mask_img": X_all, "cond": C_all}, verbose=0, batch_size=8))
    pred_vqc = _as_list(fm_vqc.predict({"mask_img": X_all, "cond": C_all}, verbose=0, batch_size=4))

    map_mlp = {ln: arr for ln, arr in zip(used_mlp, pred_mlp)}
    map_vqc = {ln: arr for ln, arr in zip(used_vqc, pred_vqc)}

    for block in BLOCK_ORDER:
        spec = BLOCK_LAYER_SPECS[block]
        mlp_layers = [ln for ln in spec["mlp"] if ln in map_mlp]
        vqc_layers = [ln for ln in spec["vqc"] if ln in map_vqc]
        if len(mlp_layers) == 0 or len(vqc_layers) == 0:
            continue

        A_4d = np.concatenate([map_mlp[ln] for ln in mlp_layers], axis=-1)
        B_4d = np.concatenate([map_vqc[ln] for ln in vqc_layers], axis=-1)

        Za = spatial_flatten(A_4d)
        Zb = spatial_flatten(B_4d)

        out_mlp = spectrum_and_metrics_from_Z(Za)
        out_vqc = spectrum_and_metrics_from_Z(Zb)

        # point estimate
        metric_rows.append({
            "seed": seed, "block": block, "model": "MLP",
            "N": A_4d.shape[0], "H": A_4d.shape[1], "W": A_4d.shape[2], "C": A_4d.shape[3],
            "M=N*H*W": Za.shape[0], "d_pr": out_mlp["d_pr"], "d_erank": out_mlp["d_erank"], "k90": out_mlp["k90"],
            "rank_eff": out_mlp["rank_eff"],
        })
        metric_rows.append({
            "seed": seed, "block": block, "model": "VQC",
            "N": B_4d.shape[0], "H": B_4d.shape[1], "W": B_4d.shape[2], "C": B_4d.shape[3],
            "M=N*H*W": Zb.shape[0], "d_pr": out_vqc["d_pr"], "d_erank": out_vqc["d_erank"], "k90": out_vqc["k90"],
            "rank_eff": out_vqc["rank_eff"],
        })

        # spectrum
        for ridx, (ev, rv, cv) in enumerate(zip(out_mlp["eigvals"], out_mlp["explained_ratio"], out_mlp["cum_explained_ratio"]), 1):
            spectra_rows.append({"seed": seed, "block": block, "model": "MLP", "rank_idx": ridx,
                                 "eigval": float(ev), "explained_ratio": float(rv), "cum_explained_ratio": float(cv)})
        for ridx, (ev, rv, cv) in enumerate(zip(out_vqc["eigvals"], out_vqc["explained_ratio"], out_vqc["cum_explained_ratio"]), 1):
            spectra_rows.append({"seed": seed, "block": block, "model": "VQC", "rank_idx": ridx,
                                 "eigval": float(ev), "explained_ratio": float(rv), "cum_explained_ratio": float(cv)})

        # inner bootstrap
        boot = inner_bootstrap_metrics(A_4d, B_4d, b_inner=B_INNER, seed=seed)
        for who in ["MLP", "VQC", "DELTA"]:
            for m in ["d_pr", "d_erank", "k90"]:
                inner_store.setdefault(block, {}).setdefault(who, {}).setdefault(m, {})[seed] = boot[who][m]

print(f"\nDone all seeds. elapsed={time.time()-t0:.1f}s")

# ========= Save: point estimates and spectra =========
metrics_df = pd.DataFrame(metric_rows)
spectra_df = pd.DataFrame(spectra_rows)

metrics_path = os.path.join(save_dir, "metrics_block_seed_model.csv")
spectra_path = os.path.join(save_dir, "spectra_long_seed_block_model.csv")
metrics_df.to_csv(metrics_path, index=False)
spectra_df.to_csv(spectra_path, index=False)

# ========= Outer hierarchical bootstrap summary (MLP/VQC/DELTA) =========
boot_rows = []
for block in sorted(inner_store.keys()):
    for who in ["MLP", "VQC", "DELTA"]:
        for m in ["d_pr", "d_erank", "k90"]:
            seed_to_arr = inner_store[block][who][m]  # {seed: inner_arr}
            mean_v, lo, hi = hierarchical_outer_ci(
                seed_to_arr, b_outer=B_OUTER, seed=OUTER_SEED, ci_level=CI_LEVEL
            )
            boot_rows.append({
                "block": block,
                "target": who,            # MLP / VQC / DELTA
                "metric": m,
                "mean": mean_v,
                f"CI{int(CI_LEVEL*100)}_low": lo,
                f"CI{int(CI_LEVEL*100)}_high": hi,
                f"significant_{int(CI_LEVEL*100)}": "Yes" if (lo > 0 or hi < 0) else "No",
                "n_seeds": len(seed_to_arr),
                "B_inner": B_INNER,
                "B_outer": B_OUTER,
            })

boot_df = pd.DataFrame(boot_rows).sort_values(["block", "target", "metric"]).reset_index(drop=True)
boot_path = os.path.join(save_dir, "bootstrap_hierarchical_block_target_metric.csv")
boot_df.to_csv(boot_path, index=False)

# ========= Common summaries =========
metrics_summary = (
    metrics_df.groupby(["block", "model"], as_index=False)
    .agg(
        d_pr_mean=("d_pr", "mean"), d_pr_std=("d_pr", "std"),
        d_erank_mean=("d_erank", "mean"), d_erank_std=("d_erank", "std"),
        k90_mean=("k90", "mean"), k90_std=("k90", "std"),
        n_seed=("seed", "nunique"),
    )
)
metrics_summary_path = os.path.join(save_dir, "metrics_block_model_summary.csv")
metrics_summary.to_csv(metrics_summary_path, index=False)

spectra_summary = (
    spectra_df.groupby(["block", "model", "rank_idx"], as_index=False)
    .agg(
        eigval_mean=("eigval", "mean"),
        eigval_std=("eigval", "std"),
        explained_ratio_mean=("explained_ratio", "mean"),
        explained_ratio_std=("explained_ratio", "std"),
        cum_explained_ratio_mean=("cum_explained_ratio", "mean"),
        cum_explained_ratio_std=("cum_explained_ratio", "std"),
    )
)
spectra_summary_path = os.path.join(save_dir, "spectra_summary_block_model_rank.csv")
spectra_summary.to_csv(spectra_summary_path, index=False)

print("\nSaved:")
print(" -", metrics_path)
print(" -", spectra_path)
print(" -", boot_path)
print(" -", metrics_summary_path)
print(" -", spectra_summary_path)

display(metrics_summary)
display(boot_df.head(30))


In [ ]:
# Load saved results and plot:
# - Row 1 (2 cols): spectrum decay + cumulative explained variance (same block)
# - Separate figure: per-block d_pr evolution with * for significance
# - Legend labels: MLP->C-UNet, VQC->C-QB-UNet

# ===== Paths =====
EXPORT_DIR = os.path.join(here, "checkpoints", "hier_boot_block_1000epochs", "_spectral_exports_bootstrap", "20260724_214915")

SPECTRA_SUMMARY_CSV = os.path.join(EXPORT_DIR, "spectra_summary_block_model_rank.csv")
METRICS_SUMMARY_CSV = os.path.join(EXPORT_DIR, "metrics_block_model_summary.csv")
BOOT_CSV = os.path.join(EXPORT_DIR, "bootstrap_hierarchical_block_target_metric.csv")

# ===== Parameters =====
PLOT_BLOCK = "bottleneck"
TOP_K = 60
METRIC_FOR_STAR = "d_pr"        # Star markers use Delta significance of d_pr
CI_COL_LOW = "CI95_low"
CI_COL_HIGH = "CI95_high"
SIG_COL = "significant_95"

BLOCK_ORDER = ["down1", "down2", "down3", "down4", "bottleneck", "up4", "up3", "up2", "up1", "uv_out"]

# Display-name mapping
NAME_MAP = {"MLP": "C-UNet", "VQC": "C-QB-UNet"}

# Colors
COLOR = {"C-UNet": "#1f77b4", "C-QB-UNet": "#d62728"}

# ===== Load data =====
spec = pd.read_csv(SPECTRA_SUMMARY_CSV)
ms = pd.read_csv(METRICS_SUMMARY_CSV)
boot = pd.read_csv(BOOT_CSV)

# Rename labels
spec["model_plot"] = spec["model"].map(NAME_MAP).fillna(spec["model"])
ms["model_plot"] = ms["model"].map(NAME_MAP).fillna(ms["model"])

# Block order
spec["block"] = pd.Categorical(spec["block"], categories=BLOCK_ORDER, ordered=True)
ms["block"] = pd.Categorical(ms["block"], categories=BLOCK_ORDER, ordered=True)

# ===== Figs 1–2: one row, two columns =====
sub = spec[(spec["block"] == PLOT_BLOCK) & (spec["rank_idx"] <= TOP_K)].copy()
sub = sub.sort_values(["model_plot", "rank_idx"])

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
ax1, ax2 = axes

# ---- Left: spectrum decay (normalized eigenvalues, log y) ----
for m in ["C-UNet", "C-QB-UNet"]:
    s = sub[sub["model_plot"] == m]
    if len(s) == 0:
        continue
    x = s["rank_idx"].values
    y = s["explained_ratio_mean"].values
    y_std = s["explained_ratio_std"].fillna(0).values
    ax1.plot(x, y, label=m, color=COLOR[m], linewidth=2)
    ax1.fill_between(x, np.clip(y-y_std, 0, 1), np.clip(y+y_std, 0, 1), alpha=0.2)

ax1.set_yscale("log")
ax1.set_xlabel("Rank index")
ax1.set_ylabel("Normalized eigenvalue")
ax1.set_title(f"Spectrum Decay ({PLOT_BLOCK})")
ax1.grid(alpha=0.25)
ax1.legend()

# ---- Right: cumulative explained variance ----
for m in ["C-UNet", "C-QB-UNet"]:
    s = sub[sub["model_plot"] == m]
    if len(s) == 0:
        continue
    x = s["rank_idx"].values
    y = s["cum_explained_ratio_mean"].values
    y_std = s["cum_explained_ratio_std"].fillna(0).values
    lo = np.clip(y - y_std, 0, 1)
    hi = np.clip(y + y_std, 0, 1)
    ax2.plot(x, y, color=COLOR[m], linewidth=2, label=m)
    ax2.fill_between(x, lo, hi, color=COLOR[m], alpha=0.2)

ax2.axhline(0.9, color="gray", linestyle="--", linewidth=1)
ax2.set_xlabel("Rank index")
ax2.set_ylabel("Cumulative explained variance")
ax2.set_title(f"Cumulative Explained Variance ({PLOT_BLOCK})")
ax2.grid(alpha=0.25)
ax2.legend()

fig.tight_layout()
plt.show()

# ===== Fig 3: per-block d_pr (+ significance stars) =====
dpr = ms[["block", "model_plot", "d_pr_mean", "d_pr_std"]].copy()
dpr = dpr.sort_values(["model_plot", "block"])

# Delta significance (DELTA rows from bootstrap)
star_df = boot[(boot["target"] == "DELTA") & (boot["metric"] == METRIC_FOR_STAR)].copy()
star_df["block"] = pd.Categorical(star_df["block"], categories=BLOCK_ORDER, ordered=True)
star_df = star_df.sort_values("block")
star_blocks = set(star_df.loc[star_df[SIG_COL] == "Yes", "block"].astype(str).tolist())

plt.figure(figsize=(8, 4))
x = np.arange(len(BLOCK_ORDER))

for m in ["C-UNet", "C-QB-UNet"]:
    s = dpr[dpr["model_plot"] == m].set_index("block").reindex(BLOCK_ORDER)
    y = s["d_pr_mean"].values.astype(float)
    y_std = s["d_pr_std"].fillna(0).values.astype(float)
    plt.plot(x, y, marker="o", linewidth=2, color=COLOR[m], label=m)
    plt.fill_between(x, y - y_std, y + y_std, color=COLOR[m], alpha=0.2)

# Star significant blocks (based on Delta)
# Star y-position: max of the two curves + small offset
s1 = dpr[dpr["model_plot"] == "C-UNet"].set_index("block").reindex(BLOCK_ORDER)["d_pr_mean"].values
s2 = dpr[dpr["model_plot"] == "C-QB-UNet"].set_index("block").reindex(BLOCK_ORDER)["d_pr_mean"].values
y_top = np.nanmax(np.vstack([s1, s2]), axis=0)
offset = 0.03 * (np.nanmax(y_top) - np.nanmin(y_top) + 1e-12)

for i, b in enumerate(BLOCK_ORDER):
    if b in star_blocks:
        plt.text(i, y_top[i] + offset, "*", ha="center", va="bottom", fontsize=16, fontweight="bold")

plt.xticks(x, BLOCK_ORDER, rotation=35, ha="right")
plt.ylabel("Participation ratio")
plt.title("Layer-wise Participation Ratio Evolution")
plt.grid(alpha=0.25)
plt.legend()
plt.tight_layout()
plt.show()

print("Significant blocks (DELTA + d_pr + 95% CI):", sorted(list(star_blocks)))
